<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/pinn_deepxde2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install deepxde
!pip install optuna

In [10]:
import os
os.environ["DDE_BACKEND"] = "pytorch"
import deepxde as dde
import numpy as np
import torch
import optuna

In [33]:
kappa = 1.0
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

def exact_solution(x):
    xx = x[:, 0:1]
    tt = x[:, 1:2]
    return np.exp(-tt) * np.sin(xx)

def objective(trial):
  iterations = trial.suggest_int("iterations", 1000, 5000, step=100)
  num_domain = trial.suggest_int("num_domain", 20, 500, step=10)

  data = dde.data.TimePDE(geomtime, pde, [], num_domain=num_domain, train_distribution="uniform", solution=exact_solution)

  layer_size = [2] + [20] * 2 + [1]
  activation = "tanh"
  initializer = "Glorot uniform"
  net = dde.nn.FNN(layer_size, activation, initializer)

  def output_transform(x, y):
      x_col = x[:, 0:1]
      t_col = x[:, 1:2]
      return torch.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

  net.apply_output_transform(output_transform)

  model = dde.Model(data, net)
  model.compile("adam", lr=1e-3, metrics=["l2 relative error"])
  losshistory, train_state = model.train(iterations=iterations)

  l2_error = float(train_state.metrics_test[-1])
  return l2_error

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print(study.best_params)
print(study.best_value)

[I 2026-11-13 06:16:11,737] A new study created in memory with name: no-name-d24784df-9ee3-49b2-8067-30839ff4ac63


Compiling model...
'compile' took 0.000180 s

Training model...

Step      Train loss    Test loss     Test metric   
0         [2.15e+00]    [2.15e+00]    [1.37e+00]    
1000      [4.02e-04]    [4.02e-04]    [8.81e-03]    
2000      [1.21e-04]    [1.21e-04]    [5.46e-03]    
3000      [5.24e-05]    [5.24e-05]    [3.46e-03]    
4000      [2.72e-05]    [2.72e-05]    [2.36e-03]    
4100      [2.58e-05]    [2.58e-05]    [2.30e-03]    

Best model at step 4100:
  train loss: 2.58e-05
  test loss: 2.58e-05
  test metric: [2.30e-03]

'train' took 19.837201 s



[I 2026-11-13 06:16:31,584] Trial 0 finished with value: 0.0023022026289254427 and parameters: {'iterations': 4100, 'num_domain': 150}. Best is trial 0 with value: 0.0023022026289254427.


Compiling model...
'compile' took 0.000177 s

Training model...

Step      Train loss    Test loss     Test metric   
0         [1.76e+00]    [1.76e+00]    [4.68e-01]    
1000      [4.49e-04]    [4.49e-04]    [7.17e-03]    
2000      [9.09e-05]    [9.09e-05]    [2.39e-03]    
2300      [6.82e-05]    [6.82e-05]    [1.94e-03]    

Best model at step 2300:
  train loss: 6.82e-05
  test loss: 6.82e-05
  test metric: [1.94e-03]

'train' took 12.695971 s



[I 2026-11-13 06:16:44,291] Trial 1 finished with value: 0.0019428397063165903 and parameters: {'iterations': 2300, 'num_domain': 450}. Best is trial 1 with value: 0.0019428397063165903.


Compiling model...
'compile' took 0.000158 s

Training model...

Step      Train loss    Test loss     Test metric   
0         [1.54e-01]    [1.54e-01]    [3.68e-01]    
1000      [4.57e-05]    [4.57e-05]    [3.26e-03]    
1200      [3.45e-05]    [3.45e-05]    [3.05e-03]    

Best model at step 1200:
  train loss: 3.45e-05
  test loss: 3.45e-05
  test metric: [3.05e-03]

'train' took 5.203729 s



... (32 outputs left out)

[I 2026-11-13 06:21:39,781] Trial 18 finished with value: 0.00281701423227787 and parameters: {'iterations': 4800, 'num_domain': 60}. Best is trial 12 with value: 0.000701748242136091.


Compiling model...
'compile' took 0.000157 s

Training model...

Step      Train loss    Test loss     Test metric   
0         [2.18e-01]    [2.18e-01]    [4.45e-01]    
1000      [2.59e-04]    [2.59e-04]    [5.53e-03]    
2000      [5.48e-05]    [5.48e-05]    [1.93e-03]    
3000      [3.13e-05]    [3.13e-05]    [1.42e-03]    
4000      [2.24e-05]    [2.24e-05]    [1.22e-03]    
4200      [2.12e-05]    [2.12e-05]    [1.21e-03]    

Best model at step 4200:
  train loss: 2.12e-05
  test loss: 2.12e-05
  test metric: [1.21e-03]

'train' took 21.446622 s



[I 2026-11-13 06:22:01,249] Trial 19 finished with value: 0.0012119857128709555 and parameters: {'iterations': 4200, 'num_domain': 280}. Best is trial 12 with value: 0.000701748242136091.


{'iterations': 4600, 'num_domain': 410}
0.000701748242136091
